# 9 案例-价格分类

学习目标

* 掌握构建分类模型流程

小明创办了一家手机公司，他不知道如何估算手机产品的价格。为了解决这个问题，他收集了多家公司的手机销售数据。

我们需要帮助小明找出手机的功能（例如：RAM等）与其售价之间的某种关系。我们可以使用机器学习的方法来解决这个问题，也可以构建一个全连接的网络。

需要注意的是：在这个问题中，我们不需要预测实际价格，而是一个价格范围，它的范围使用 0、1、2、3 来表示，所以该问题也是一个分类问题。

## 1. 构建数据集

数据共有 2000 条，其中 1600 条数据作为训练集，400 条数据用作测试集。我们使用 sklearn 的数据集划分工作来完成。并使用 PyTorch 的 TensorDataset 来将数据集构建为 Dataset 对象，方便构造数据加载加载对象。

In [ ]:
# 构建数据集
def create_dataset():

    data = pd.read_csv('data/手机价格预测.csv')

    # 特征值和目标值
    x, y = data.iloc[:, :-1], data.iloc[:, -1]
    x = x.astype(np.float32)
    y = y.astype(np.int64)

    # 数据集划分
    x_train, x_valid, y_train, y_valid = \
        train_test_split(x, y, train_size=0.8, random_state=88, stratify=y)

    # 构建数据集å
    train_dataset = TensorDataset(torch.from_numpy(x_train.values), torch.tensor(y_train.values))
    valid_dataset = TensorDataset(torch.from_numpy(x_valid.values), torch.tensor(y_valid.values))

    return train_dataset, valid_dataset, x_train.shape[1], len(np.unique(y))


train_dataset, valid_dataset, input_dim, class_num = create_dataset()

* `train_test_split` 是 scikit-learn 库中用于将数据集分割为训练集和测试集的函数。下面我将详细解释你提到的每个参数的含义：

* **x**: 这是输入特征的数组或矩阵。通常，`x` 包含你想要用于训练模型的所有特征数据。

* **y**: 这是目标变量的数组。`y` 包含与 `x` 中的每个样本相对应的标签或目标值。

* **train_size**: 这个参数指定训练集的比例。在这个例子中，`train_size=0.8` 意味着 80% 的数据将被用作训练集，剩下的 20% 将用作测试集。

* **random_state**: 这个参数用于设置随机数生成器的种子，确保每次运行代码时分割的结果都是一致的。在这个例子中，`random_state=88` 意味着使用种子 88 来初始化随机数生成器。

* **stratify**: 这个参数用于分层抽样。在这个例子中，`stratify=y` 意味着训练集和测试集中的目标变量 `y` 的类别分布将与原始数据集中的类别分布保持一致。这对于确保训练集和测试集具有相似的类别比例非常有用，特别是在处理不平衡数据集时。

## 2. 构建分类网络模型

我们构建的用于手机价格分类的模型叫做全连接神经网络。它主要由三个线性层来构建，在每个线性层后，我们使用的是 sigmoid 激活函数。

In [ ]:
# 构建网络模型
class PhonePriceModel(nn.Module):

    def __init__(self, input_dim, output_dim):
        super(PhonePriceModel, self).__init__()

        self.linear1 = nn.Linear(input_dim, 128)
        self.linear2 = nn.Linear(128, 256)
        self.linear3 = nn.Linear(256, output_dim)

    def _activation(self, x):
        return torch.sigmoid(x)

    def forward(self, x):

        x = self._activation(self.linear1(x))
        x = self._activation(self.linear2(x))
        output = self.linear3(x)

        return output

我们的网络共有 3 个全连接层，具体信息如下：

1. 第一层：输入维度为 20，输出维度为：128
2. 第二层：输入维度为 128，输出维度为：256
3. 第三层：输入维度为 256，输出维度为：4

我们使用 sigmoid 激活函数。

## 3. 编写训练函数

网络编写完成之后，我们需要编写训练函数。所谓的训练函数，指的是输入数据读取、送入网络、计算损失、更新参数的流程，该流程较为固定。我们使用的是多分类交叉熵损失函数、使用 SGD 优化方法。最终，将训练好的模型持久化到磁盘中。

## 3. 编写训练函数

网络编写完成之后，我们需要编写训练函数。所谓的训练函数，指的是输入数据读取、送入网络、计算损失、更新参数的流程，该流程较为固定。我们使用的是多分类交叉熵损失函数、使用 SGD 优化方法。最终，将训练好的模型持久化到磁盘中。

In [ ]:
def train():

    # 固定随机数种子
    torch.manual_seed(0)

    # 初始化模型
    model = PhonePriceModel(input_dim, class_num)
    # 损失函数
    criterion = nn.CrossEntropyLoss()
    # 优化方法
    optimizer = optim.SGD(model.parameters(), lr=1e-3)
    # 训练轮数
    num_epoch = 50

    for epoch_idx in range(num_epoch):

        # 初始化数据加载器
        dataloader = DataLoader(train_dataset, shuffle=True, batch_size=8)
        # 训练时间
        start = time.time()
        # 计算损失
        total_loss = 0.0
        total_num = 1
        # 准确率
        correct = 0

        for x, y in dataloader:

            output = model(x)
            # 计算损失
            loss = criterion(output, y)
            # 梯度清零
            optimizer.zero_grad()
            # 反向传播
            loss.backward()
            # 参数更新
            optimizer.step()

            total_num += len(y)
            total_loss += loss.item() * len(y)

        print('epoch: %4s loss: %.2f, time: %.2fs' % \
            (epoch_idx + 1, total_loss / total_num, time.time() - start))

    # 模型保存
    torch.save(model.state_dict(), 'model/phone-price-model.bin')

## 4. 编写评估函数

评估函数、也叫预测函数、推理函数，主要使用训练好的模型，对未知的样本的进行预测的过程。我们这里使用前面单独划分出来的测试集来进行评估。

In [ ]:
def test():

    # 加载模型
    model = PhonePriceModel(input_dim, class_num)
    model.load_state_dict(torch.load('model/phone-price-model.bin'))

    # 构建加载器
    dataloader = DataLoader(valid_dataset, batch_size=8, shuffle=False)

    # 评估测试集
    correct = 0
    for x, y in dataloader:

        output = model(x)
        y_pred = torch.argmax(output, dim=1)
        correct += (y_pred == y).sum()

    print('Acc: %.5f' % (correct.item() / len(valid_dataset)))

## 5. 网络性能调优

我们前面的网络模型在测试集的准确率为：0.54750，我们可以通过以下方面进行调优：

1. 对输入数据进行标准化
2. 调整优化方法
3. 调整学习率
4. 增加批量归一化层
5. 增加网络层数、神经元个数
6. 增加训练轮数
7. 等等...

我进行下如下调整：1. 优化方法由 SGD 调整为 Adam 2. 学习率由 1e-3 调整为 1e-4 3. 对数据数据进行标准化 4. 增加网络深度，即：增加网络参数数量

网络模型在测试集的准确率由 0.5475 上升到 0.9625，调整后的完整代码为：

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
import torch.optim as optim
import numpy as np
import time
from sklearn.preprocessing import StandardScaler

# 构建数据集
def create_dataset():

    data = pd.read_csv('data/手机价格预测.csv')

    # 特征值和目标值
    x, y = data.iloc[:, :-1], data.iloc[:, -1]
    x = x.astype(np.float32)
    y = y.astype(np.int64)

    # 数据集划分
    x_train, x_valid, y_train, y_valid = \
        train_test_split(x, y, train_size=0.8, random_state=88, stratify=y)

    # 构建数据集å
    train_dataset = TensorDataset(torch.from_numpy(x_train.values), torch.tensor(y_train.values))
    valid_dataset = TensorDataset(torch.from_numpy(x_valid.values), torch.tensor(y_valid.values))

    return train_dataset, valid_dataset, x_train.shape[1], len(np.unique(y))


train_dataset, valid_dataset, input_dim, class_num = create_dataset()

```python
# 构建网络模型
class PhonePriceModel(nn.Module):

    def __init__(self, input_dim, output_dim):
        super(PhonePriceModel, self).__init__()

        self.linear1 = nn.Linear(input_dim, 128)
        self.linear2 = nn.Linear(128, 256)
        self.linear3 = nn.Linear(256, 512)
        self.linear4 = nn.Linear(512, 128)
        self.linear5 = nn.Linear(128, output_dim)

    def _activation(self, x):
        return torch.sigmoid(x)

    def forward(self, x):

        x = self._activation(self.linear1(x))
        x = self._activation(self.linear2(x))
        x = self._activation(self.linear3(x))
        x = self._activation(self.linear4(x))
        output = self.linear5(x)

        return output
    
```python
# 编写训练函数
def train():

    # 固定随机数种子
    torch.manual_seed(0)

    # 初始化模型
    model = PhonePriceModel(input_dim, class_num)
    # 损失函数
    criterion = nn.CrossEntropyLoss()
    # 优化方法
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    # 训练轮数
    num_epoch = 50

    for epoch_idx in range(num_epoch):

        # 初始化数据加载器
        dataloader = DataLoader(train_dataset, shuffle=True, batch_size=8)
        # 训练时间
        start = time.time()
        # 计算损失
        total_loss = 0.0
        total_num = 1
        # 准确率
        correct = 0

        for x, y in dataloader:

            output = model(x)
            # 计算损失
            loss = criterion(output, y)
            # 梯度清零
            optimizer.zero_grad()
            # 反向传播
            loss.backward()
            # 参数更新
            optimizer.step()

            total_num += len(y)
            total_loss += loss.item() * len(y)

        print('epoch: %4s loss: %.2f, time: %.2fs' % \
            (epoch_idx + 1, total_loss / total_num, time.time() - start))

    # 模型保存
    torch.save(model.state_dict(), 'model/phone-price-model.bin')


def test():

    # 加载模型
    model = PhonePriceModel(input_dim, class_num)
    model.load_state_dict(torch.load('model/phone-price-model.bin'))

    # 构建加载器
    dataloader = DataLoader(valid_dataset, batch_size=8, shuffle=False)

    # 评估测试集
    correct = 0
    for x, y in dataloader:

        output = model(x)
        y_pred = torch.argmax(output, dim=1)
        correct += (y_pred == y).sum()

    print('Acc: %.5f' % (correct.item() / len(valid_dataset)))

if __name__ == '__main__':
    train()
    test()